# Multithreading

## 📑 Table of Contents

**1. Foundations**
- [🎯 Learning Goal](#learning-goal)
- [🤔 What is it?](#what-is-it)
- [❓ Why do we need it?](#why-do-we-need-it)
- [🧠 Key Idea](#key-idea)
- [📚 Important Terms](#important-terms)
- [🔄 How it Works](#how-it-works)
- [🌍 Real-Life Example](#real-life-example)

**2. Process vs Thread**
- [⚙️ Process vs Thread](#process-vs-thread)

**3. Creating and Managing Threads**
- [🧵 Creating a Thread — the `target` Way](#creating-threads)
- [🏗️ Creating a Thread — Subclassing `Thread`](#subclassing-thread)
- [▶️ `start()` vs `run()`](#start-vs-run)

**4. The GIL, CPU-Bound vs I/O-Bound**
- [🔒 The GIL (Global Interpreter Lock)](#the-gil)
- [🖥️ CPU-Bound vs I/O-Bound Tasks](#cpu-vs-io-bound)
- [⚡ `multiprocessing` — True Parallelism for CPU-Bound Work](#multiprocessing)
- [🔢 A Bigger Example — 40 Lakh Heavy Multiplications](#lakh-example)
- [🧭 Decision Guide — Sequential vs Threading vs Multiprocessing](#decision-guide)

**5. Thread Safety**
- [⚠️ Race Conditions](#race-conditions)
- [🔐 `Lock` — Fixing Race Conditions](#locks)
- [🚦 `Semaphore` — Limiting Concurrent Access](#semaphores)
- [🚩 `Event` — Signaling Between Threads](#events)
- [🗄️ `threading.local()` — Per-Thread Data](#thread-local)

**6. Daemon Threads**
- [👻 Daemon Threads](#daemon-threads)

**7. Practical Patterns**
- [🏭 Producer-Consumer with `queue.Queue`](#producer-consumer)
- [🧰 `ThreadPoolExecutor` — Managing a Pool of Threads](#thread-pool-executor)
- [🧭 When to Use (and When NOT to Use) Threading](#when-to-use-threading)

**8. Multithreading Across Languages**
- [🌐 Comparison with Other Languages](#language-comparison)

**9. Wrap-Up**
- [⚖ Comparison Tables](#comparison)
- [⚠ Common Misconceptions](#misconceptions)
- [🔍 Interview Questions](#interview-questions)
- [📝 Quick Revision](#quick-revision)
- [🎓 Cheat Sheet](#cheat-sheet)
- [📖 Related Topics](#related-topics)
- [✅ Key Takeaways](#key-takeaways)

> 💡 Click any link above to jump straight to that section.

<a id="learning-goal"></a>
## 🎯 Learning Goal

By the end of this note, I will understand what a thread is and how it differs from a process, how to create and manage threads in Python with the `threading` module, what the GIL is and why it limits CPU-bound multithreading, when to use threads vs `multiprocessing`, how race conditions happen and how to fix them (`Lock`, `Semaphore`, `Event`), daemon threads, practical patterns (`Queue`, `ThreadPoolExecutor`), and how Python's threading model compares to Java, C++, JavaScript/Node.js, and Go.

---

<a id="what-is-it"></a>
## 🤔 What is it?

A **thread** is the smallest unit of execution inside a program — a single sequential flow of instructions. **Multithreading** means running several threads at once, INSIDE the same process, so they can share memory and work on different parts of a problem seemingly at the same time.

> 🧸 Think of a process like a restaurant kitchen, and threads like the chefs working in it. All the chefs (threads) share the SAME kitchen — same fridge, same counters, same ingredients (memory) — which makes handing things off between them fast, but also means two chefs reaching for the same knife at the same time can cause a collision (a **race condition**, covered later).

---

<a id="why-do-we-need-it"></a>
## ❓ Why do we need it?

- Many programs spend most of their time **waiting** — for a network response, a file read, a database query — and a single thread just sits idle during that wait. Another thread can use that idle time productively.
- Some tasks are naturally independent — downloading 5 files, handling 100 web requests — and running them concurrently instead of one-by-one can be dramatically faster.
- Without threads, a slow operation (like a network call) would freeze an ENTIRE program — including things like a responsive UI — until it finishes.

---

<a id="key-idea"></a>
## 🧠 Key Idea

- A **process** has its own private memory; **threads** inside the same process SHARE that memory — this makes threads lightweight, but also makes shared data dangerous to modify without care.
- Python's **GIL (Global Interpreter Lock)** allows only ONE thread to execute Python bytecode at a time — so threading gives a real speedup for **I/O-bound** work (waiting), but NOT for **CPU-bound** work (heavy computation).
- For CPU-bound parallelism, Python reaches for `multiprocessing` instead — separate processes, separate memory, separate GILs, genuinely running at the same time on multiple CPU cores.
- When multiple threads modify shared data without coordination, you get a **race condition** — the fix is synchronization primitives like `Lock`, `Semaphore`, and `Event`.
- A **daemon thread** is a background thread that Python will kill automatically when the main program exits, without waiting for it to finish.

---

<a id="important-terms"></a>
## 📚 Important Terms

| Term | Simple Meaning | Example |
|------|----------------|----------|
| Process | An independent running program with its own memory | Each open Chrome tab (roughly) |
| Thread | A single flow of execution inside a process | `threading.Thread(target=func)` |
| Concurrency | Multiple tasks making progress, possibly by interleaving (not necessarily at the same instant) | Threads on a single CPU core |
| Parallelism | Multiple tasks running at the EXACT same instant, on different cores | Multiple `multiprocessing.Process` |
| GIL | Global Interpreter Lock — only one thread runs Python bytecode at a time (CPython) | Why CPU-bound threading doesn't speed things up |
| Race Condition | A bug from multiple threads reading/writing shared data without coordination | Two threads both reading a stale `balance` |
| Lock | A synchronization tool ensuring only ONE thread enters a section of code at a time | `with lock:` |
| Daemon Thread | A background thread killed automatically when the main program exits | `Thread(daemon=True)` |
| CPU-Bound | Limited by processor speed (heavy computation) | Calculating primes, image processing |
| I/O-Bound | Limited by waiting on external operations | Network calls, disk reads, `time.sleep()` |

---

<a id="how-it-works"></a>
## 🔄 How it Works

```mermaid
flowchart TD
Main["Main Thread"] -->|Thread(target=f)| T1["Thread 1 created"]
Main -->|Thread(target=g)| T2["Thread 2 created"]
T1 -->|.start()| Run1["Thread 1 runs concurrently"]
T2 -->|.start()| Run2["Thread 2 runs concurrently"]
Run1 -->|.join()| Wait["Main thread waits for both to finish"]
Run2 -->|.join()| Wait
Wait --> Done["Main thread continues"]
```

Read it like this: the main thread creates worker threads and calls `.start()` on each — they then run concurrently, interleaved by the GIL (on CPython) rather than truly simultaneously for CPU work. `.join()` tells the main thread to pause and wait until a given thread has actually finished, before continuing.

---

<a id="real-life-example"></a>
## 🌍 Real-Life Example

Think of a **restaurant with one shared kitchen** (a process) and **several chefs** (threads) working in it. If a chef is waiting for water to boil (an I/O-bound wait — like a network call), another chef can use the stove in the meantime — genuinely useful overlap. But if the kitchen only has ONE cutting board that must be shared (like the GIL only allowing one thread to run Python code at a time), only one chef can actually be chopping vegetables (executing Python bytecode) at any given instant — everyone else has to wait their turn, even if they all "seem" to be working at once.

---

<a id="process-vs-thread"></a>
## ⚙️ Process vs Thread

A **process** is an independent, running instance of a program — it has its OWN memory space, its own resources, and is isolated from other processes by the operating system. A **thread** is a unit of execution WITHIN a process — multiple threads in the same process share that process's memory.

| | Process | Thread |
|---|---|---|
| Memory | Own, isolated memory space | Shares memory with other threads in the same process |
| Creation cost | Expensive (OS sets up a whole new memory space) | Cheap (lightweight, reuses the process's memory) |
| Communication | Needs explicit IPC (pipes, queues, shared memory) | Direct — just shared variables (careful: race conditions!) |
| Crash impact | One process crashing does NOT crash another | One thread crashing an unhandled exception can affect the whole process |
| Parallelism (Python) | TRUE parallel execution — each process has its own GIL | Limited by the GIL — only I/O-bound work benefits |
| Real-world analogy | Separate restaurant kitchens, each with their own equipment | Chefs sharing ONE kitchen and its equipment |

> 🧸 If a process is a house, a thread is a person living in it. Multiple people (threads) in the same house (process) can walk into the same kitchen and grab the same knife (shared memory) — fast and convenient, but they can bump into each other. Two separate houses (processes) never share a kitchen at all — safer, but you'd need to physically carry things between houses (inter-process communication) to share anything.

### 🔍 Interview Questions — Process vs Thread

- What is the fundamental difference between a process and a thread?
- Why is creating a new thread generally cheaper than creating a new process?
- If one thread crashes with an unhandled exception, what happens to the other threads in the same process? What about if one PROCESS crashes?
- Why do threads need synchronization (locks) when accessing shared data, but processes usually don't (for their own memory)?
- In Python specifically, why would you choose `multiprocessing` over `threading` for a CPU-heavy task?
- How do processes typically communicate with each other, since they don't share memory?

---

<a id="creating-threads"></a>
## 🧵 Creating a Thread — the `target` Way

`threading.Thread(target=function, args=(...))` is the simplest way to create a thread — hand it a function to run and the arguments to call it with.

In [2]:
import threading
import time

def print_numbers():
    for i in range(1, 4):
        time.sleep(0.05)
        print(f"Number: {i}")

def print_letters():
    for letter in "ABC":
        time.sleep(0.05)
        print(f"Letter: {letter}")


t1 = threading.Thread(target=print_numbers)
t2 = threading.Thread(target=print_letters)

t1.start()   # starts running CONCURRENTLY -- doesn't wait for t1 to finish before moving on
t2.start()

t1.join()    # NOW wait here until t1 actually finishes
t2.join()    # and wait here until t2 actually finishes

print("Both threads finished")


Number: 1Letter: A

Letter: B
Number: 2
Letter: C
Number: 3
Both threads finished


**Note:** Notice the output INTERLEAVES "Number:" and "Letter:" lines — both threads are genuinely running concurrently, not one after another. The exact interleaving order can vary between runs (that's the nature of concurrency) — but "Both threads finished" always prints LAST, because `join()` blocks the main thread until each worker thread is actually done.

> ❌ Misconception: `t1.start()` runs the thread and waits for it to complete before moving to the next line.
> ✅ Correct: `start()` KICKS OFF the thread and returns immediately — the main thread keeps going right away. `join()` is the one that actually waits.

---

<a id="subclassing-thread"></a>
## 🏗️ Creating a Thread — Subclassing `Thread`

For more complex threads that need their own state, you can subclass `threading.Thread` and override its `run()` method — this is the object-oriented alternative to the `target=` approach.

In [3]:
class CounterThread(threading.Thread):
    def __init__(self, name, count):
        super().__init__()          # must call the parent's __init__ to set up the thread properly
        self.thread_name = name
        self.count = count

    def run(self):                    # this is what executes when start() is called
        for i in range(1, self.count + 1):
            print(f"{self.thread_name}: {i}")


t1 = CounterThread("Thread-A", 3)
t2 = CounterThread("Thread-B", 3)
t1.start()
t2.start()
t1.join()
t2.join()
print("Done")


Thread-A: 1Thread-B: 1
Thread-B: 2
Thread-B: 3

Thread-A: 2
Thread-A: 3
Done


**Note:** Subclassing is preferred when a thread needs its own persistent state (like `self.count` here) or more complex setup logic — `super().__init__()` is required, exactly like any other subclass in Python OOP, so the parent `Thread` class can initialize its internal bookkeeping.

---

<a id="start-vs-run"></a>
## ▶️ `start()` vs `run()`

`start()` and `run()` sound like they might do the same thing — they don't, and mixing them up is a classic beginner mistake.

In [4]:
def worker():
    print("Running inside:", threading.current_thread().name)

# WRONG -- calling run() directly just calls it like a normal function, on the CURRENT thread
t1 = threading.Thread(target=worker)
print("Calling run() directly (WRONG):")
t1.run()
print("Is a new thread alive?", t1.is_alive())   # False -- no new thread was ever created

print("---")

# RIGHT -- start() actually spawns a new thread, which then calls run() internally
t2 = threading.Thread(target=worker)
t2.start()
t2.join()


Calling run() directly (WRONG):
Running inside: MainThread
Is a new thread alive? False
---
Running inside: Thread-10 (worker)


**Note:** Calling `t1.run()` directly executes `worker()` like a plain function call, on whatever thread called it (`MainThread`) — `is_alive()` correctly reports `False` afterward, because no new thread was ever actually created. Only `start()` asks the operating system to spin up a real new thread, which THEN internally calls `run()` for you. This is exactly analogous to calling `Car.start_engine(car)` vs. actually turning the key — `run()` is the engine's internals, `start()` is what actually kicks off the whole process.

---

<a id="the-gil"></a>
## 🔒 The GIL (Global Interpreter Lock)

The **GIL** is a lock inside CPython (the standard Python interpreter) that allows only ONE thread to execute Python bytecode at any given instant — even on a multi-core machine, even with many threads created.

> 🧸 Imagine a busy kitchen with 4 chefs (threads) but only ONE knife (the GIL) in the entire kitchen. Chefs can still take turns quickly — chopping a little, passing the knife, someone else chops a little — which LOOKS like everyone's cutting vegetables "at once" if you glance quickly. But strictly speaking, only ONE chef is ever actually cutting at any single moment.

### ❓ Why does the GIL exist? (The Use Case)

CPython manages memory using **reference counting** — every Python object carries a counter of how many things point to it, and gets freed the instant that counter hits zero. This counter gets incremented/decremented CONSTANTLY, on nearly every line of Python code (assigning a variable, passing an argument, appending to a list all touch it).

If two threads updated the SAME object's reference count at the exact same instant, the update could be lost (this is a race condition — see [Race Conditions](#race-conditions) below) — leading to memory being freed too early (a crash) or never freed at all (a leak). The GIL's job is to make this reference counting safe with minimal effort: by only ever letting ONE thread touch Python objects at a time, no two threads can corrupt a reference count simultaneously.

The GIL was also a deliberate, pragmatic trade-off: it made CPython's core interpreter simpler to write and dramatically easier for C extension authors (`numpy`, `pandas`'s internals, etc.) to write thread-unsafe C code without worrying about Python-level concurrency — a huge reason CPython's C-extension ecosystem became so rich. Removing the GIL entirely requires a much more complex, fine-grained locking scheme (per-object locks instead of one global lock), which historically made single-threaded code slower — a bad trade-off for the vast majority of Python programs that aren't heavily multithreaded. (Newer CPython versions have an experimental "free-threaded" build without a GIL, but it isn't the default yet.)

### ⚙️ Why Can't Python Threads Run Python Code Truly in Parallel?

Mechanically, here's what actually happens: whichever thread currently HOLDS the GIL is the only one allowed to execute Python bytecode. Every so often (by a configurable interval, or when it hits a blocking I/O call), the running thread is forced to RELEASE the GIL, and the interpreter lets another waiting thread acquire it and run for a while. This creates the ILLUSION of parallelism through fast switching — but at any single instant, there is only ever ONE thread actually executing Python instructions, no matter how many CPU cores the machine has.

This is exactly why:
- **I/O-bound work releases the GIL early** (the moment a thread starts waiting on `time.sleep()`, a network call, or a file read, CPython releases the GIL immediately, instead of waiting for the usual switch interval) — so OTHER threads get to run productively during that wait. This is genuine, useful concurrency.
- **CPU-bound work never voluntarily releases the GIL** — it's constantly executing bytecode with nothing to wait on, so threads just take turns in short bursts, achieving no real speedup over running sequentially (and paying a small cost for all that switching).
- **`multiprocessing` sidesteps the GIL entirely** — because each `Process` gets its OWN Python interpreter and its OWN separate GIL, multiple processes really can execute Python bytecode at the exact same instant, on different CPU cores. This is the only way to get TRUE CPU-bound parallelism in standard CPython.

<a id="cpu-vs-io-bound"></a>
### 🖥️ CPU-Bound vs I/O-Bound Tasks

This is THE key distinction for knowing whether threading will actually help:

| | CPU-Bound | I/O-Bound |
|---|---|---|
| Bottleneck | The processor itself — heavy calculation | Waiting on something EXTERNAL (network, disk, another system) |
| Examples | Number crunching, image processing, sorting huge lists, cryptography | Network requests, reading/writing files, database queries, `time.sleep()` |
| Does the GIL block progress? | ✅ Yes — only one thread computes at a time regardless of thread count | ❌ No — a waiting thread RELEASES the GIL, letting others run |
| Does `threading` help? | ❌ Barely, if at all — often the SAME speed (or slower) than sequential | ✅ Yes — real, significant speedup |
| What actually helps | `multiprocessing` (separate processes, separate GILs, real parallel cores) | `threading` (or `asyncio`) is perfect here |

Let's prove this with two matching examples: identical work, but CPU-bound vs I/O-bound.

In [5]:
def cpu_bound_task(n):
    count = 0
    for i in range(n):
        count += i * i     # pure computation -- the CPU is always busy, never waiting
    return count


N = 20_000_000

# Sequential baseline -- run the CPU-bound task twice, one after another
start = time.time()
cpu_bound_task(N)
cpu_bound_task(N)
sequential_time = time.time() - start
print(f"Sequential (2 CPU-bound tasks): {sequential_time:.2f}s")

# Threaded -- run the SAME two tasks, but on two separate threads
start = time.time()
t1 = threading.Thread(target=cpu_bound_task, args=(N,))
t2 = threading.Thread(target=cpu_bound_task, args=(N,))
t1.start(); t2.start()
t1.join(); t2.join()
threaded_time = time.time() - start
print(f"Threaded (2 CPU-bound tasks):   {threaded_time:.2f}s   <-- barely faster, sometimes even slower!")


Sequential (2 CPU-bound tasks): 1.42s
Threaded (2 CPU-bound tasks):   1.39s   <-- barely faster, sometimes even slower!


In [6]:
def io_bound_task():
    time.sleep(0.2)   # simulates waiting on a network request or disk read -- NOT actual CPU work


# Sequential -- one waits, THEN the next waits
start = time.time()
for _ in range(4):
    io_bound_task()
sequential_time = time.time() - start
print(f"Sequential (4 I/O-bound tasks): {sequential_time:.2f}s")

# Threaded -- all 4 wait AT THE SAME TIME, because waiting releases the GIL
start = time.time()
threads = [threading.Thread(target=io_bound_task) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
threaded_time = time.time() - start
print(f"Threaded (4 I/O-bound tasks):   {threaded_time:.2f}s   <-- roughly 4x faster!")


Sequential (4 I/O-bound tasks): 0.82s
Threaded (4 I/O-bound tasks):   0.21s   <-- roughly 4x faster!


**Note:** These two results tell the whole story. `time.sleep()` (and real I/O like network/disk waits) RELEASES the GIL while waiting, letting other threads run during that idle time — so 4 threads that each "wait" for 0.2s finish in about 0.2s total, not 0.8s. But `cpu_bound_task()` never releases the GIL for long (it's constantly executing bytecode) — so two threads doing CPU work take roughly the SAME total time as doing them one after another, because they're really just taking turns on the one "knife," not truly working in parallel.

> ❌ Misconception: adding more threads always makes a Python program faster.
> ✅ Correct: it only helps for I/O-bound work. For CPU-bound work, threading in Python (CPython) provides little to no speedup — and can even be slightly SLOWER, due to the overhead of constantly switching which thread holds the GIL.

---

<a id="multiprocessing"></a>
## ⚡ `multiprocessing` — True Parallelism for CPU-Bound Work

`multiprocessing.Process` creates a genuinely separate PROCESS, with its OWN Python interpreter and its OWN GIL — so multiple processes really CAN run Python bytecode simultaneously, on different CPU cores.

In [7]:
import subprocess

# multiprocessing.Process on some platforms (e.g. macOS) uses the "spawn" start method, which re-imports
# the target function in a fresh interpreter -- this only works reliably for functions defined in a real
# .py file, not ones defined inline in a notebook cell. So we write a small script and run it as a
# separate process, exactly like the __name__ == "__main__" module demo from Part 1 of this series.
with open("mp_demo.py", "w") as f:
    f.write('''import multiprocessing
import time

def cpu_bound_task(n):
    count = 0
    for i in range(n):
        count += i * i
    return count

if __name__ == "__main__":
    N = 20_000_000
    start = time.time()
    p1 = multiprocessing.Process(target=cpu_bound_task, args=(N,))
    p2 = multiprocessing.Process(target=cpu_bound_task, args=(N,))
    p1.start(); p2.start()
    p1.join(); p2.join()
    print(f"Multiprocessing (2 CPU-bound tasks): {time.time() - start:.2f}s   <-- genuinely faster, real parallel cores!")
''')

result = subprocess.run(["python3", "mp_demo.py"], capture_output=True, text=True)
print(result.stdout.strip())


Multiprocessing (2 CPU-bound tasks): 0.67s   <-- genuinely faster, real parallel cores!


**Note:** Compare this runtime to the "Threaded (2 CPU-bound tasks)" result above — `multiprocessing` achieves a REAL speedup for the exact same computation, because each `Process` runs in its own interpreter with its own GIL, genuinely executing on separate CPU cores at the same time. The trade-off: processes are heavier to create than threads, and they don't share memory directly — passing data between them requires explicit mechanisms (`multiprocessing.Queue`, `Pipe`, shared memory), unlike threads which share memory for free (at the cost of needing locks to do it safely).

> ❌ Misconception: `multiprocessing` is just a "better" or "faster" version of `threading` you should always prefer.
> ✅ Correct: it solves a DIFFERENT problem — real parallelism for CPU-bound work, at the cost of higher memory usage and slower inter-process communication. For I/O-bound work, `threading` (or `asyncio`) is usually simpler and just as fast.

<a id="lakh-example"></a>
### 🔢 A Bigger Example — 40 Lakh (4,000,000) Heavy Multiplications

Let's prove the speedup with a genuinely heavy CPU-bound task at real scale: **40 lakh (4,000,000) multiplications**, split across 4 workers — run three ways (sequential, threading, multiprocessing) so the difference is unmistakable.

In [ ]:
import subprocess

# Same reason as the earlier multiprocessing demo -- Process needs real functions from a real .py file
# on platforms using the "spawn" start method, so this runs as a separate script.
with open("lakh_multiply_demo.py", "w") as f:
    f.write('''import multiprocessing
import threading
import time

def heavy_multiply_range(start_end):
    start, end = start_end
    total = 0
    for i in range(start, end):
        result = i
        for _ in range(10):          # repeat the multiplication a few times -- simulates a HEAVIER task
            result = result * i
        total += result
    return total

if __name__ == "__main__":
    N = 4_000_000   # 40 lakh
    num_workers = 4
    chunk_size = N // num_workers
    chunks = [(i * chunk_size, (i + 1) * chunk_size) for i in range(num_workers)]

    # ---- 1. Sequential -- one process does all 40 lakh multiplications, no splitting ----
    start = time.time()
    sequential_total = heavy_multiply_range((0, N))
    seq_time = time.time() - start
    print(f"Sequential  (1 worker,  {N:,} multiplications): {seq_time:.2f}s")

    # ---- 2. Threading -- split across 4 THREADS ----
    thread_results = [None] * num_workers
    def thread_worker(idx, chunk):
        thread_results[idx] = heavy_multiply_range(chunk)

    start = time.time()
    threads = [threading.Thread(target=thread_worker, args=(i, chunks[i])) for i in range(num_workers)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    thread_time = time.time() - start
    print(f"Threading   ({num_workers} threads, {N:,} multiplications): {thread_time:.2f}s")

    # ---- 3. Multiprocessing -- split across 4 PROCESSES ----
    start = time.time()
    with multiprocessing.Pool(processes=num_workers) as pool:
        pool.map(heavy_multiply_range, chunks)
    mp_time = time.time() - start
    print(f"Multiproc.  ({num_workers} procs,   {N:,} multiplications): {mp_time:.2f}s")

    print()
    print(f"Threading speedup vs sequential:      {seq_time / thread_time:.2f}x")
    print(f"Multiprocessing speedup vs sequential: {seq_time / mp_time:.2f}x")
''')

result = subprocess.run(["python3", "lakh_multiply_demo.py"], capture_output=True, text=True)
print(result.stdout.strip())


**Note:** With 40 lakh heavy multiplications split 4 ways: **Threading finishes in almost the SAME time as sequential** (roughly `1.0x` — no real speedup at all, exactly as the GIL predicts), while **Multiprocessing finishes in roughly a THIRD of the time** (close to `3x`, out of an ideal `4x` for 4 workers — the gap from perfect is the overhead of starting 4 separate processes and combining their results). This is the clearest possible proof: for a genuinely heavy, CPU-bound task, `threading` does nothing for you, and `multiprocessing` delivers a real, measurable win.

<a id="decision-guide"></a>
### 🧭 Decision Guide — Sequential vs Threading vs Multiprocessing

| | Sequential | Threading | Multiprocessing |
|---|---|---|---|
| **Use when** | The task is small/fast enough that concurrency isn't worth the complexity | The task is **I/O-bound** — waiting on network, disk, or another system | The task is **CPU-bound** — heavy computation with no waiting |
| **Why it works (or doesn't)** | N/A — no concurrency, simplest to reason about | Waiting RELEASES the GIL, so other threads make progress during the wait | Each process has its OWN GIL — genuinely runs on separate CPU cores at once |
| **Speedup on heavy computation** | Baseline (1x) | ~1x — little to no real speedup (GIL blocks parallel computation) | Up to ~Nx with N cores (real speedup, as shown above) |
| **Speedup on I/O-bound waiting** | Baseline (1x) | Up to ~Nx — genuinely effective | Also effective, but usually overkill — more memory/startup cost for no extra benefit |
| **Memory cost** | Lowest | Low — threads share the process's memory | Higher — each process gets its own full memory space |
| **Setup/creation cost** | None | Cheap | Expensive (new interpreter per process) |
| **Data sharing between workers** | N/A | Direct (shared variables) — needs `Lock`/`Semaphore` for safety | Explicit only (`Queue`, `Pipe`, shared memory) — no free sharing |
| **Typical real examples** | A simple script, a one-off calculation | Downloading multiple files, calling several APIs, reading multiple files | Image/video processing, large numeric simulations, data crunching over big arrays |

> 🧸 The short version: **waiting → `threading`**, **computing → `multiprocessing`**, **neither → plain sequential code, don't overcomplicate it**.

---

<a id="race-conditions"></a>
## ⚠️ Race Conditions

A **race condition** happens when multiple threads read and write SHARED data without coordination, and the final result depends on the unpredictable TIMING of which thread does what, when. The classic example: a bank balance being read, modified, and written back — with another thread sneaking in during that gap.

In [8]:
balance = 100

def withdraw(amount):
    global balance
    current = balance          # READ the current balance
    time.sleep(0.01)             # simulate some processing time -- forces a context switch HERE
    balance = current - amount    # WRITE, based on a now possibly-STALE value of "current"


# Two threads, each trying to withdraw 50 from a balance of 100 -- expected final balance: 0
threads = [threading.Thread(target=withdraw, args=(50,)) for _ in range(2)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("Expected: 0, Got:", balance)


Expected: 0, Got: 50


**Bug found:** the output prints `Got: 50`, not the expected `0`. Here's exactly what happened: BOTH threads read `balance` as `100` before EITHER of them got to subtract anything (because `time.sleep(0.01)` forces a switch right after the read). Both threads then independently compute `100 - 50 = 50` and write `50` back — the second write just OVERWRITES the first, and one entire withdrawal is silently lost. This is a real, dangerous class of bug precisely because it's **non-deterministic** — it might not show up every run, making it notoriously hard to catch in testing.

> ❌ Misconception: `global balance` and simple `+=`/`-=` operations are automatically thread-safe in Python.
> ✅ Correct: even something as simple as `balance = current - amount` is actually multiple separate steps (read, compute, write) — and ANY of those steps can be interrupted by another thread. Nothing about Python's syntax makes an operation atomic just because it looks like "one line."

---

<a id="locks"></a>
## 🔐 `Lock` — Fixing Race Conditions

`threading.Lock()` ensures only ONE thread can execute a given block of code at a time — any other thread trying to enter has to wait until the lock is released.

In [17]:
balance = 1000
lock = threading.Lock()

def withdraw_safe(amount):
    global balance
    with lock:                  # only ONE thread can be inside this block at a time -- others wait here
        current = balance
        time.sleep(0.01)
        balance = current - amount


threads = [threading.Thread(target=withdraw_safe, args=(100,)) for _ in range(2)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("Expected: 800, Got:", balance)


Expected: 800, Got: 800


**Note:** `with lock:` is the idiomatic way to use a `Lock` — it calls `lock.acquire()` on entry and `lock.release()` on exit AUTOMATICALLY, even if an exception happens inside the block (the exact same pattern as `with open(...) as f:` guaranteeing `close()`). Now the SECOND thread to reach `with lock:` must wait until the FIRST one finishes its entire read-sleep-write sequence — no more overlapping, no more lost updates. This correctly prints `0` every single time.

> ❌ Misconception: you should just wrap EVERYTHING in locks to be safe.
> ✅ Correct: locks make code SLOWER (threads spend time waiting instead of working) and can cause **deadlocks** if used incorrectly (e.g., two threads each holding a lock the other one needs). Only protect the SPECIFIC shared data that's actually being modified by multiple threads — not entire functions "just in case."

---

<a id="semaphores"></a>
## 🚦 `Semaphore` — Limiting Concurrent Access

A `Lock` allows exactly ONE thread in at a time. A `Semaphore` generalizes this to allow up to N threads in at once — useful for limiting access to a resource that can handle SOME concurrency, just not unlimited concurrency (e.g., a pool of 2 database connections).

In [10]:
semaphore = threading.Semaphore(2)   # only 2 threads may hold this at the same time

def access_resource(name):
    with semaphore:
        print(f"{name} acquired the resource")
        time.sleep(0.1)
        print(f"{name} released the resource")


threads = [threading.Thread(target=access_resource, args=(f"Worker-{i}",)) for i in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()


Worker-0 acquired the resource
Worker-1 acquired the resource
Worker-1 released the resource
Worker-2 acquired the resource
Worker-0 released the resource
Worker-3 acquired the resource
Worker-2 released the resourceWorker-3 released the resource



**Note:** With 4 workers but a `Semaphore(2)`, only 2 workers can be "inside" (holding the resource) at any given moment — the other 2 wait until one of the first two releases it. Watch the output: you'll always see at most 2 "acquired" lines before the first "released" line appears. A `Lock` is really just `Semaphore(1)` — a special case with exactly one slot.

---

<a id="events"></a>
## 🚩 `Event` — Signaling Between Threads

A `threading.Event` is a simple flag one thread can WAIT on, and another thread can SET — useful for "wait until something happens elsewhere" coordination, without locks or shared counters.

In [11]:
event = threading.Event()

def waiter():
    print("Waiter: waiting for the signal...")
    event.wait()   # BLOCKS here until event.set() is called by some other thread
    print("Waiter: got the signal, proceeding!")

def setter():
    time.sleep(0.1)
    print("Setter: sending the signal now")
    event.set()


t1 = threading.Thread(target=waiter)
t2 = threading.Thread(target=setter)
t1.start()
t2.start()
t1.join()
t2.join()


Waiter: waiting for the signal...
Setter: sending the signal now
Waiter: got the signal, proceeding!


**Note:** "Waiter: waiting..." always prints FIRST, and "Waiter: got the signal!" always prints LAST, no matter how the two threads happen to be scheduled — `event.wait()` genuinely blocks until `event.set()` is called somewhere else. This is a cleaner alternative to manually polling a shared boolean variable in a loop (`while not ready: pass`), which would waste CPU cycles constantly checking.

---

<a id="thread-local"></a>
## 🗄️ `threading.local()` — Per-Thread Data

Normally, all threads share the same global variables. `threading.local()` creates an object where EACH thread automatically gets its OWN separate copy of any attribute you set on it — genuinely private, per-thread storage.

In [12]:
local_data = threading.local()

def show_value():
    local_data.value = threading.current_thread().name   # each thread sets its OWN "value"
    print(f"{threading.current_thread().name} sees local_data.value = {local_data.value}")


threads = [threading.Thread(target=show_value) for _ in range(3)]
for t in threads:
    t.start()
for t in threads:
    t.join()


Thread-27 (show_value) sees local_data.value = Thread-27 (show_value)
Thread-28 (show_value) sees local_data.value = Thread-28 (show_value)
Thread-29 (show_value) sees local_data.value = Thread-29 (show_value)


**Note:** All 3 threads run the exact SAME function (`show_value`) and set the SAME attribute name (`local_data.value`), yet each one sees only ITS OWN value — never another thread's. `threading.local()` is commonly used for things like per-thread database connections or per-request context in web servers, where you want isolation without manually passing extra parameters through every function call.

---

<a id="daemon-threads"></a>
## 👻 Daemon Threads

A **daemon thread** runs in the background and is automatically killed the instant the main program exits — Python does NOT wait for daemon threads to finish, unlike normal ("non-daemon") threads.

In [13]:
def background_worker():
    counter = 0
    while True:                # an infinite loop -- would run FOREVER on its own
        counter += 1
        time.sleep(0.01)


t = threading.Thread(target=background_worker, daemon=True)   # daemon=True set at creation time
t.start()
print("Daemon thread running:", t.is_alive())
print("Is it marked as a daemon?", t.daemon)

time.sleep(0.05)
print("Main program finishing -- the daemon thread is killed automatically, no join() needed")


Daemon thread running: True
Is it marked as a daemon? True
Main program finishing -- the daemon thread is killed automatically, no join() needed


**Note:** `background_worker()` has a genuinely infinite `while True:` loop — if this were a NORMAL (non-daemon) thread, the whole program would HANG forever waiting for it (or at `join()`, if called). Because `daemon=True` was set, Python is free to just terminate it the instant the main program (here, the notebook cell) finishes — no cleanup code inside the daemon thread runs, it's simply cut off. This is exactly why daemon threads are only appropriate for background work where an abrupt, uncontrolled stop is genuinely fine — a periodic "heartbeat" log, a cache-cleanup loop — never for anything that must finish cleanly (like flushing a file buffer or completing a database write).

> ❌ Misconception: daemon threads are a way to run background tasks that finish "eventually," even after the main program ends.
> ✅ Correct: it's the opposite — daemon threads are terminated abruptly WHEN the main program ends, with no guarantee they get to finish (or even clean up) their current work at all.

---

<a id="producer-consumer"></a>
## 🏭 Producer-Consumer with `queue.Queue`

The **producer-consumer pattern** has one or more threads PRODUCING items and one or more threads CONSUMING them, coordinated through a shared queue. `queue.Queue` is already internally thread-safe — no manual `Lock` needed.

In [14]:
import queue

q = queue.Queue()

def producer():
    for i in range(5):
        q.put(i)                    # add an item to the queue -- thread-safe, no lock needed
        print(f"Produced: {i}")
        time.sleep(0.01)

def consumer():
    for _ in range(5):
        item = q.get()               # blocks here until an item is available
        print(f"Consumed: {item}")
        q.task_done()                 # signal that this item has been fully processed


t1 = threading.Thread(target=producer)
t2 = threading.Thread(target=consumer)
t1.start()
t2.start()
t1.join()
t2.join()
print("All items processed")


Produced: 0
Consumed: 0
Produced: 1Consumed: 1

Produced: 2Consumed: 2

Produced: 3
Consumed: 3
Produced: 4
Consumed: 4
All items processed


**Note:** `q.get()` automatically BLOCKS the consumer if the queue is currently empty, waiting until the producer adds something — no manual `Event` or polling loop required. `queue.Queue` handles all the internal locking for you, which is why it's the recommended way to pass data between threads, instead of a plain shared `list` (which is NOT safely shared without your own locking).

> ❌ Misconception: any built-in Python collection (`list`, `dict`) is safe to share between threads without extra care.
> ✅ Correct: `queue.Queue` is SPECIFICALLY designed and internally locked for cross-thread use. Plain `list`/`dict` operations are usually "safe enough" for simple appends due to the GIL, but compound operations (check-then-modify) are NOT safe — `queue.Queue` removes the guesswork entirely.

---

<a id="thread-pool-executor"></a>
## 🧰 `ThreadPoolExecutor` — Managing a Pool of Threads

Manually creating and joining a `Thread` for every single task gets unwieldy fast. `concurrent.futures.ThreadPoolExecutor` manages a reusable POOL of worker threads for you, and hands tasks to whichever thread is free.

In [15]:
from concurrent.futures import ThreadPoolExecutor

def fetch_data(url_id):
    time.sleep(0.1)   # simulate a slow network call
    return f"Data from url {url_id}"


with ThreadPoolExecutor(max_workers=3) as executor:   # only 3 threads active at once, reused across tasks
    results = list(executor.map(fetch_data, [1, 2, 3, 4, 5]))

for r in results:
    print(r)


Data from url 1
Data from url 2
Data from url 3
Data from url 4
Data from url 5


**Note:** `executor.map(fetch_data, [1, 2, 3, 4, 5])` schedules all 5 calls, but only 3 run CONCURRENTLY at any moment (`max_workers=3`) — as soon as one finishes, the pool picks up the next pending task, reusing the same 3 threads rather than creating 5 separate ones. `results` comes back in the SAME order the inputs were given, regardless of which thread finished first — `map()` handles that ordering for you. The `with` block ensures every thread in the pool is properly cleaned up when it exits, just like closing a file.

`ThreadPoolExecutor` is almost always preferred over manually managing `Thread` objects for anything beyond a couple of one-off threads — it's less code, avoids accidentally creating too many threads at once, and its API (`submit()`, `map()`, `Future` objects) is shared with `ProcessPoolExecutor`, making it easy to switch between threads and processes later.

---

<a id="when-to-use-threading"></a>
## 🧭 When to Use (and When NOT to Use) Threading in Python

Given everything above about the GIL, here's the practical decision guide.

| ✅ Use Threading When... | ❌ Avoid Threading When... |
|---|---|
| The task is **I/O-bound** — network requests, API calls, web scraping | The task is **CPU-bound** — heavy computation, number crunching, image/video processing |
| Reading/writing **multiple files** concurrently | You need **true parallel execution** across CPU cores — use `multiprocessing` instead |
| Making **many concurrent network calls** (hitting several APIs/URLs at once) | The work is so **small/fast** that thread creation overhead outweighs any benefit |
| Keeping a **UI responsive** while a slow operation runs in the background | The logic is **simple and sequential** with no real waiting — threading only adds complexity for nothing |
| Handling **many simultaneous connections** (a simple socket server, polling multiple sources) | You need **guaranteed completion** of the work — daemon threads can be killed mid-task; use non-daemon threads with proper `join()`, or a process |
| Waiting on **user input** alongside other background work | The shared state is **complex and heavily mutated** — many locks quickly become hard to reason about and prone to deadlocks; consider `queue.Queue`-based message passing, `multiprocessing`, or `asyncio` instead |
| Overlapping **database queries** or **disk reads** that mostly wait on external systems | You specifically need **isolation** between tasks (one crashing shouldn't affect another) — use separate processes instead |

> 🧸 The one-question shortcut: **"Is this task waiting on something outside the CPU, or is it making the CPU work hard?"** Waiting (I/O-bound) → threading helps. Working hard (CPU-bound) → threading won't help; reach for `multiprocessing`.

**A quick self-check before reaching for threads at all:** could this be simpler as a single sequential function? If the "concurrency" only saves a trivial amount of time, or the task runs once and finishes quickly, the added complexity (thread safety, debugging non-deterministic bugs) usually isn't worth it.

---

<a id="language-comparison"></a>
## 🌐 Comparison with Other Languages

Python's GIL is NOT how every language handles multithreading — this is one of the biggest surprises for developers coming from (or going to) another language.

| | Python (CPython) | Java | C++ | JavaScript / Node.js | Go |
|---|---|---|---|---|---|
| Has a GIL-like lock? | ✅ Yes — the GIL | ❌ No | ❌ No | N/A — single-threaded event loop model | ❌ No |
| True CPU-bound parallelism with threads? | ❌ No (use `multiprocessing` instead) | ✅ Yes — real OS threads, real parallel cores | ✅ Yes — `std::thread`, real parallel cores | ❌ No — JS itself runs on ONE thread (the event loop); true parallel CPU work needs Worker Threads (separate V8 instances) | ✅ Yes — goroutines scheduled across OS threads |
| Default concurrency unit | OS thread (`threading.Thread`) or process (`multiprocessing.Process`) | OS thread (`Thread`, or virtual "Project Loom" threads in newer versions) | OS thread (`std::thread`) | Callback / Promise / `async`-`await` on ONE thread, plus a libuv thread pool for I/O | Goroutine — a lightweight, GO-managed "green thread" |
| Shared-memory safety | Manual (`Lock`, `Semaphore`, etc.) — GIL helps somewhat with simple ops, but NOT compound ones | Manual (`synchronized`, `Lock`, `java.util.concurrent`) | Manual (`std::mutex`, `std::atomic`) | Rarely an issue for CPU work (single-threaded); Worker Threads need their own manual coordination | Manual, but idiomatically via CHANNELS (`chan`) instead of shared memory + locks |
| Cost of creating a unit of concurrency | Thread: cheap-ish; Process: expensive | Thread: moderate; Loom virtual threads: very cheap | Thread: moderate (OS-level) | Very cheap (just a callback/promise) | Extremely cheap (goroutines start around a few KB of stack) |
| Best "escape hatch" for CPU-bound work | `multiprocessing` (separate processes) | Just use more threads — no GIL to work around | Just use more threads — no GIL to work around | Worker Threads, or offload to another process/service | Just use more goroutines — no GIL to work around |

> 🧸 Java, C++, and Go give every thread a REAL seat at the CPU-core table — if you have 4 cores, 4 threads can genuinely compute at once. Python's GIL is like a single conference room badge that only one thread can hold — great for avoiding chaos in a small office, but a real bottleneck when everyone actually needs to work at once. JavaScript takes a totally different approach: it doesn't even TRY to give you multiple workers by default — it's one very fast, very organized single worker who never blocks (thanks to the event loop), delegating slow I/O work to the "kitchen staff" behind the scenes (libuv's thread pool) and picking the results back up later.

### 🔍 Key Differences Worth Knowing for Interviews

- **Java** threads are OS-level threads with true parallelism — Java's own concurrency utilities (`ExecutorService`, `synchronized`, `java.util.concurrent.locks`) look conceptually similar to Python's `ThreadPoolExecutor`/`Lock`, but Java threads actually achieve CPU-bound speedups that Python threads cannot.
- **C++** gives you the most low-level, manual control (`std::thread`, `std::mutex`, `std::atomic`) with zero built-in safety net — the programmer is fully responsible for correctness, and there's no GIL cushioning simple mistakes at all.
- **JavaScript/Node.js** is fundamentally single-threaded for your OWN code — concurrency comes from the event loop (non-blocking I/O via callbacks/promises/`async`-`await`), NOT from multiple threads running your JS simultaneously. True parallel CPU work requires explicitly spinning up Worker Threads, each with its OWN separate JS engine instance (conceptually similar to Python's `multiprocessing`, not `threading`).
- **Go** was designed around concurrency from day one: goroutines are extremely cheap (you can spawn thousands with ease) and are scheduled by the Go runtime across available OS threads and CPU cores automatically — no GIL, and the idiomatic way to share data is "don't share memory, communicate instead" via channels, rather than locking shared variables.
- Python's `multiprocessing` is the closest equivalent to "just use threads normally" in Java/C++/Go for CPU-bound work — but with the added overhead of separate memory spaces, since Python can't give threads true parallelism the way those languages can.

---

<a id="comparison"></a>
## ⚖ Comparison Tables

| | `threading.Thread` | `multiprocessing.Process` |
|---|---|---|
| Memory | Shared with other threads | Own, isolated memory |
| Best for | I/O-bound work | CPU-bound work |
| Overhead | Low | Higher (new interpreter per process) |
| Communication | Direct (shared variables + locks) | Explicit (`Queue`, `Pipe`) |

| | `Lock` | `Semaphore` | `Event` |
|---|---|---|---|
| Purpose | Only ONE thread in a section at a time | Up to N threads in a section at a time | Signal "something happened" to waiting threads |
| Typical use | Protecting a shared counter/balance | Limiting connections to a resource pool | "Wait until setup is done" coordination |

| | Manual `Thread` management | `ThreadPoolExecutor` |
|---|---|---|
| Boilerplate | You create, start, and join every thread | Pool handles creation/reuse/cleanup |
| Risk of too many threads | Higher (easy to spawn unbounded threads) | Lower (`max_workers` caps concurrency) |
| Getting return values back | Manual (e.g., via a shared list + lock) | Built-in (`Future` objects, `map()` return values) |

---

<a id="misconceptions"></a>
## ⚠ Common Misconceptions

❌ Multithreading in Python always makes your program run faster.
✅ Only for I/O-bound work. For CPU-bound work, the GIL means threading provides little to no speedup — use `multiprocessing` instead.

❌ `t.start()` runs a thread's code immediately and waits for it to finish.
✅ `start()` returns immediately after kicking off the thread; `join()` is what actually waits for it to finish.

❌ Calling `t.run()` starts a new thread, just like `t.start()`.
✅ `run()` just calls the target function like a normal method call, on the current thread — no new thread is created.

❌ Simple operations like `counter += 1` are automatically thread-safe in Python.
✅ Even simple-looking operations are multiple steps (read, modify, write) internally, and can be interrupted by another thread mid-operation, causing a race condition.

❌ You should wrap everything in locks "just to be safe."
✅ Locks add overhead and can cause deadlocks. Only protect the specific shared data that multiple threads actually modify.

❌ Daemon threads keep running in the background even after the main program exits, until they finish naturally.
✅ The opposite — daemon threads are killed abruptly the moment the main program exits, with no guarantee they finish their current work.

❌ Python's GIL is a universal feature of "multithreading" in every language.
✅ It's specific to CPython. Java, C++, and Go give threads true parallel execution; JavaScript's single-threaded event loop model is a completely different approach again.

---

<a id="interview-questions"></a>
## 🔍 Interview Questions

- What is the GIL, and why does it exist in CPython?
- Why does multithreading help I/O-bound tasks but not CPU-bound tasks in Python?
- What's the difference between `threading` and `multiprocessing` in Python — when would you use each?
- What is a race condition? Walk through a concrete example where one occurs.
- How does a `Lock` fix a race condition? What's the risk of using locks incorrectly?
- What's the difference between `Lock` and `Semaphore`?
- What does `threading.Event` do, and when would you use it over a `Lock`?
- What is `threading.local()` used for?
- What is a daemon thread, and how does it differ from a normal thread?
- Why is `queue.Queue` preferred over a plain `list` for passing data between threads?
- What does `ThreadPoolExecutor` give you that manually managing `Thread` objects doesn't?
- How does Python's threading model differ from Java's or Go's? Why can Java/Go threads achieve real CPU-bound parallelism when Python's can't?
- If you needed to process 1,000 independent CPU-heavy calculations as fast as possible in Python, would you reach for `threading` or `multiprocessing`? Why?
- What's the difference between concurrency and parallelism?

---

<a id="quick-revision"></a>
## 📝 Quick Revision

- A process has its own memory; threads inside a process SHARE memory — threads are cheaper to create but need coordination.
- `threading.Thread(target=func)` or subclassing `Thread` (overriding `run()`) both create threads; `start()` actually launches them, `join()` waits for them to finish.
- `start()` ≠ `run()` — calling `run()` directly just calls the function normally, with no new thread.
- The GIL lets only ONE thread execute Python bytecode at a time in CPython — I/O-bound work releases the GIL while waiting (real speedup from threading); CPU-bound work doesn't (no real speedup).
- `multiprocessing.Process` sidesteps the GIL entirely by using separate processes with separate interpreters — the right tool for CPU-bound parallelism.
- A race condition happens when threads read/write shared data without coordination — fix it with `Lock` (one at a time), `Semaphore` (up to N at a time), or restructure to avoid sharing mutable state.
- `threading.Event` signals "something happened" between threads without polling; `threading.local()` gives each thread its own private copy of data.
- Daemon threads (`daemon=True`) are killed abruptly when the main program exits — only use them for background work that's fine to interrupt.
- `queue.Queue` is internally thread-safe and is the standard way to pass data between producer and consumer threads.
- `ThreadPoolExecutor` manages a reusable pool of threads, handling creation, task scheduling, and cleanup — usually preferred over manual `Thread` management.
- Java, C++, and Go give threads TRUE parallel execution (no GIL); JavaScript/Node.js is single-threaded by default, achieving concurrency through an event loop instead of OS threads.

---

<a id="cheat-sheet"></a>
## 🎓 Cheat Sheet

| Concept | One-Line Meaning |
|----------|------------------|
| `threading.Thread(target=f)` | Create a thread that will run `f` |
| `t.start()` | Actually launch the thread (returns immediately) |
| `t.join()` | Block until the thread finishes |
| `t.run()` | ⚠️ Just calls the function directly — does NOT create a thread |
| GIL | Only one thread executes Python bytecode at a time (CPython) |
| CPU-bound | Threading barely helps — use `multiprocessing` |
| I/O-bound | Threading helps a lot — waiting releases the GIL |
| Race condition | Bug from unsynchronized shared-data access across threads |
| `threading.Lock()` | Only ONE thread in a protected section at a time |
| `threading.Semaphore(n)` | Up to `n` threads in a protected section at a time |
| `threading.Event()` | Signal / wait-for-signal coordination between threads |
| `threading.local()` | Per-thread private data |
| `daemon=True` | Thread is killed abruptly when the main program exits |
| `queue.Queue` | Thread-safe queue — standard producer-consumer tool |
| `ThreadPoolExecutor` | Managed, reusable pool of worker threads |
| `multiprocessing.Process` | Separate process, separate GIL — true CPU parallelism |

---

<a id="related-topics"></a>
## 📖 Related Topics

Since this topic is **Multithreading**, next recommended topics:

- [Exception Handling](24_ExceptionHandling.ipynb) (unhandled exceptions inside a thread don't crash the main program — they just terminate that thread silently unless handled)
- [File Handling](25_FileHandling.ipynb) (file I/O is a classic I/O-bound task that benefits from threading)
- `multiprocessing` in depth (`Pool`, `Queue`, `Pipe`, shared memory via `Value`/`Array`)
- `asyncio` — Python's other concurrency model, built around a single-threaded event loop (closer to JavaScript's approach) rather than OS threads
- [OOP in Python — Part 1](21_Oops_part1.ipynb) and [Part 2](22_Oops_part2.ipynb) (subclassing `Thread` uses the same inheritance and `super().__init__()` patterns)

---

<a id="key-takeaways"></a>
## ✅ Key Takeaways

1. A process has isolated memory; threads share memory within a process — sharing is fast but requires careful coordination.
2. `start()` launches a new thread and returns immediately; `join()` blocks until it finishes; calling `run()` directly does NOT create a thread at all.
3. The GIL lets only one thread execute Python bytecode at a time — threading gives real speedups for I/O-bound work (waiting releases the GIL) but little to none for CPU-bound work.
4. `multiprocessing` achieves true parallelism for CPU-bound work by using separate processes, each with its own interpreter and GIL — at the cost of higher memory use and explicit inter-process communication.
5. Race conditions happen when multiple threads read/write shared data without coordination — `Lock` and `Semaphore` fix this by controlling how many threads can enter a critical section at once.
6. `Event` (signaling) and `threading.local()` (per-thread data) solve coordination problems that locks alone don't address well.
7. Daemon threads are killed abruptly when the main program exits — only appropriate for background work that's safe to interrupt.
8. `queue.Queue` and `ThreadPoolExecutor` are the practical, production-ready tools for most real threading use cases — preferred over manually managing raw `Thread` objects and shared collections.
9. Python's GIL is a CPython-specific design choice, not a universal law of multithreading — Java, C++, and Go give threads true parallel execution, while JavaScript/Node.js takes a different single-threaded event-loop approach entirely.